In [ ]:
%matplotlib inline
import numpy as np
import scipy.signal as scipy_sig
import matplotlib.pyplot as plt
import soundfile as sf
import json
from pathlib import Path
from abc import ABC, abstractmethod
from IPython.display import display, Audio
import ipywidgets as widgets

BASE_DIR = Path('.')
RAW_DIR = BASE_DIR / 'data' / 'raw'
PROCESSED_DIR = BASE_DIR / 'data' / 'processed'
PRESET_DIR = BASE_DIR / 'data' / 'presets'
for d in [RAW_DIR, PROCESSED_DIR, PRESET_DIR]:
    d.mkdir(parents=True, exist_ok=True)
SR = 44100
print('Imports OK')

## Section 1: Audio Effects Classes

We reproduce all effect classes here so notebook 03 is fully self-contained.

In [ ]:
class AudioEffect(ABC):
    def __init__(self, name='effect'):
        self.name = name
        self.bypass = False
    @abstractmethod
    def process(self, signal, sample_rate): pass
    def get_params(self):
        return {k: v for k, v in self.__dict__.items()
                if not k.startswith('_') and k not in ('name',) and not callable(v)
                and not isinstance(v, np.ndarray)}
    def set_params(self, **kwargs):
        for k, v in kwargs.items():
            if hasattr(self, k): setattr(self, k, v)

class SchroederReverb(AudioEffect):
    def __init__(self, room_size=0.5, damping=0.5, wet_mix=0.5, pre_delay_ms=20.0):
        super().__init__('schroeder_reverb')
        self.room_size = float(np.clip(room_size, 0.1, 1.0))
        self.damping = float(np.clip(damping, 0.0, 1.0))
        self.wet_mix = float(wet_mix)
        self.pre_delay_ms = float(pre_delay_ms)
    def _comb(self, x, d, fb, dp):
        d = max(1, int(d)); buf = np.zeros(d); y = np.zeros(len(x)); fs = 0.0; pos = 0
        d1=dp; d2=1-dp
        for i in range(len(x)):
            o=buf[pos]; fs=o*d2+fs*d1; buf[pos]=x[i]+fs*fb; y[i]=o; pos=(pos+1)%d
        return y
    def _ap(self, x, d, g=0.5):
        d=max(1,int(d)); buf=np.zeros(d); y=np.zeros(len(x)); pos=0
        for i in range(len(x)):
            b=buf[pos]; buf[pos]=x[i]+b*g; y[i]=b-x[i]*g; pos=(pos+1)%d
        return y
    def process(self, signal, sample_rate):
        dry = signal.astype(np.float32)
        pd = int(self.pre_delay_ms * 0.001 * sample_rate)
        x = np.concatenate([np.zeros(pd), dry])[:len(dry)]
        sc = 0.5 + self.room_size * 0.5
        cds = [int(d*sc*sample_rate/1000) for d in [29.7,37.1,41.1,43.7]]
        fb = 0.7 + self.room_size * 0.18
        co = sum(self._comb(x,cd,fb,self.damping*0.4) for cd in cds) * 0.25
        for ad in [int(5.0*sample_rate/1000), int(1.7*sample_rate/1000)]:
            co = self._ap(co, ad)
        out = (1-self.wet_mix)*dry + self.wet_mix*co
        return np.clip(out,-1,1).astype(np.float32)

class DigitalDelay(AudioEffect):
    def __init__(self, delay_ms=300.0, feedback=0.4, wet_mix=0.4):
        super().__init__('digital_delay')
        self.delay_ms = float(delay_ms)
        self.feedback = float(np.clip(feedback, 0.0, 0.99))
        self.wet_mix = float(wet_mix)
    def process(self, signal, sample_rate):
        dry = signal.astype(np.float32)
        ds = max(1, int(self.delay_ms * sample_rate / 1000))
        dl = np.zeros(ds); wet = np.zeros_like(dry); pos = 0
        for i in range(len(dry)):
            d = dl[pos]; wet[i] = d
            dl[pos] = dry[i] + d * self.feedback; pos = (pos+1) % ds
        return np.clip((1-self.wet_mix)*dry + self.wet_mix*wet, -1, 1).astype(np.float32)

class Distortion(AudioEffect):
    def __init__(self, drive=5.0, mode='soft', pre_gain=1.0, post_gain=0.5, wet_mix=1.0):
        super().__init__('distortion')
        self.drive = float(max(1.0, drive))
        self.mode = mode
        self.pre_gain = float(pre_gain)
        self.post_gain = float(post_gain)
        self.wet_mix = float(wet_mix)
    def process(self, signal, sample_rate):
        dry = signal.astype(np.float32)
        x = dry * self.pre_gain * self.drive
        if self.mode == 'soft': wet = np.tanh(x)
        elif self.mode == 'hard': wet = np.clip(x, -1, 1)
        else: wet = np.where(x>=0, np.tanh(x), np.tanh(x*0.5)*1.5)
        return np.clip((1-self.wet_mix)*dry + self.wet_mix*wet*self.post_gain, -1, 1).astype(np.float32)

class Compressor(AudioEffect):
    def __init__(self, threshold_db=-20.0, ratio=4.0, attack_ms=10.0,
                 release_ms=100.0, knee_db=6.0, makeup_gain_db=6.0):
        super().__init__('compressor')
        self.threshold_db = float(threshold_db)
        self.ratio = float(max(1.0, ratio))
        self.attack_ms = float(attack_ms)
        self.release_ms = float(release_ms)
        self.knee_db = float(knee_db)
        self.makeup_gain_db = float(makeup_gain_db)
    def process(self, signal, sample_rate):
        x = signal.astype(np.float64)
        ac = np.exp(-1/(self.attack_ms*0.001*sample_rate))
        rc = np.exp(-1/(self.release_ms*0.001*sample_rate))
        T=self.threshold_db; R=self.ratio; W=self.knee_db
        mk = 10**(self.makeup_gain_db/20)
        level_db = 20*np.log10(np.abs(x)+1e-10)
        gdb = np.zeros_like(level_db)
        for i in range(len(level_db)):
            lv = level_db[i]
            if 2*(lv-T)<-W: gdb[i]=0
            elif 2*abs(lv-T)<=W: gdb[i]=(1/R-1)*(lv-T+W/2)**2/(2*W)
            else: gdb[i]=(1/R-1)*(lv-T)
        sg = np.zeros_like(gdb); prev=0.0
        for i in range(len(gdb)):
            sg[i] = ac*prev+(1-ac)*gdb[i] if gdb[i]<prev else rc*prev+(1-rc)*gdb[i]
            prev=sg[i]
        return np.clip(x*10**(sg/20)*mk,-1,1).astype(np.float32)

class ParametricEQ(AudioEffect):
    def __init__(self, low_shelf_freq=80.0, low_shelf_gain_db=0.0,
                 peak1_freq=250.0, peak1_gain_db=0.0, peak1_q=1.0,
                 peak2_freq=1000.0, peak2_gain_db=0.0, peak2_q=1.0,
                 peak3_freq=4000.0, peak3_gain_db=0.0, peak3_q=1.0,
                 high_shelf_freq=8000.0, high_shelf_gain_db=0.0):
        super().__init__('parametric_eq')
        self.low_shelf_freq=float(low_shelf_freq); self.low_shelf_gain_db=float(low_shelf_gain_db)
        self.peak1_freq=float(peak1_freq); self.peak1_gain_db=float(peak1_gain_db); self.peak1_q=float(peak1_q)
        self.peak2_freq=float(peak2_freq); self.peak2_gain_db=float(peak2_gain_db); self.peak2_q=float(peak2_q)
        self.peak3_freq=float(peak3_freq); self.peak3_gain_db=float(peak3_gain_db); self.peak3_q=float(peak3_q)
        self.high_shelf_freq=float(high_shelf_freq); self.high_shelf_gain_db=float(high_shelf_gain_db)
    def _ls(self, f, g, sr):
        A=10**(g/40); w0=2*np.pi*f/sr
        alpha=np.sin(w0)/2*np.sqrt((A+1/A)*(1/0.707-1)+2)
        b0=A*((A+1)-(A-1)*np.cos(w0)+2*np.sqrt(A)*alpha)
        b1=2*A*((A-1)-(A+1)*np.cos(w0))
        b2=A*((A+1)-(A-1)*np.cos(w0)-2*np.sqrt(A)*alpha)
        a0=(A+1)+(A-1)*np.cos(w0)+2*np.sqrt(A)*alpha
        a1=-2*((A-1)+(A+1)*np.cos(w0)); a2=(A+1)+(A-1)*np.cos(w0)-2*np.sqrt(A)*alpha
        return [b0/a0,b1/a0,b2/a0],[1,a1/a0,a2/a0]
    def _hs(self, f, g, sr):
        A=10**(g/40); w0=2*np.pi*f/sr
        alpha=np.sin(w0)/2*np.sqrt((A+1/A)*(1/0.707-1)+2)
        b0=A*((A+1)+(A-1)*np.cos(w0)+2*np.sqrt(A)*alpha)
        b1=-2*A*((A-1)+(A+1)*np.cos(w0))
        b2=A*((A+1)+(A-1)*np.cos(w0)-2*np.sqrt(A)*alpha)
        a0=(A+1)-(A-1)*np.cos(w0)+2*np.sqrt(A)*alpha
        a1=2*((A-1)-(A+1)*np.cos(w0)); a2=(A+1)-(A-1)*np.cos(w0)-2*np.sqrt(A)*alpha
        return [b0/a0,b1/a0,b2/a0],[1,a1/a0,a2/a0]
    def _pk(self, f, g, Q, sr):
        A=10**(g/40); w0=2*np.pi*f/sr; alpha=np.sin(w0)/(2*Q)
        b0=1+alpha*A; b1=-2*np.cos(w0); b2=1-alpha*A
        a0=1+alpha/A; a1=-2*np.cos(w0); a2=1-alpha/A
        return [b0/a0,b1/a0,b2/a0],[1,a1/a0,a2/a0]
    def process(self, signal, sample_rate):
        x = signal.astype(np.float32)
        for b,a in [self._ls(self.low_shelf_freq,self.low_shelf_gain_db,sample_rate),
                    self._pk(self.peak1_freq,self.peak1_gain_db,self.peak1_q,sample_rate),
                    self._pk(self.peak2_freq,self.peak2_gain_db,self.peak2_q,sample_rate),
                    self._pk(self.peak3_freq,self.peak3_gain_db,self.peak3_q,sample_rate),
                    self._hs(self.high_shelf_freq,self.high_shelf_gain_db,sample_rate)]:
            x = scipy_sig.lfilter(b,a,x).astype(np.float32)
        return np.clip(x,-1,1).astype(np.float32)

class BitCrusher(AudioEffect):
    def __init__(self, bit_depth=8, downsample=4, wet_mix=1.0):
        super().__init__('bit_crusher')
        self.bit_depth=max(1,min(16,int(bit_depth)))
        self.downsample=max(1,int(downsample))
        self.wet_mix=float(wet_mix)
    def process(self, signal, sample_rate):
        dry=signal.astype(np.float32)
        held=dry[(np.arange(len(dry))//self.downsample)*self.downsample] if self.downsample>1 else dry.copy()
        lvls=2**self.bit_depth
        q=np.round(held*(lvls/2))/(lvls/2)
        return np.clip((1-self.wet_mix)*dry+self.wet_mix*q,-1,1).astype(np.float32)

class Chorus(AudioEffect):
    def __init__(self, num_voices=3, rate_hz=1.5, depth_ms=7.0, delay_ms=20.0, wet_mix=0.5):
        super().__init__('chorus')
        self.num_voices=max(2,min(4,int(num_voices)))
        self.rate_hz=float(rate_hz); self.depth_ms=float(depth_ms)
        self.delay_ms=float(delay_ms); self.wet_mix=float(wet_mix)
    def process(self, signal, sample_rate):
        dry=signal.astype(np.float32); n=len(dry)
        t=np.arange(n)/sample_rate
        md=int((self.delay_ms+self.depth_ms)*sample_rate/1000)+2
        padded=np.concatenate([np.zeros(md),dry])
        wet=np.zeros(n)
        for v in range(self.num_voices):
            ph=2*np.pi*v/self.num_voices
            lfo=np.sin(2*np.pi*self.rate_hz*(1+0.1*v)*t+ph)
            ds=(self.delay_ms+self.depth_ms*lfo)*sample_rate/1000
            for i in range(n):
                di=int(ds[i]); fr=ds[i]-di
                idx=md+i-di
                if 1<=idx<len(padded):
                    wet[i]+=padded[idx]*(1-fr)+padded[max(0,idx-1)]*fr
        wet/=self.num_voices
        return np.clip((1-self.wet_mix)*dry+self.wet_mix*wet,-1,1).astype(np.float32)

print('All effect classes defined.')

## Section 2: EffectsChain Class

The `EffectsChain` manages an ordered sequence of `AudioEffect` instances. Each effect can be individually bypassed. The chain is processed linearly: the output of each effect is the input to the next.

In [ ]:
class EffectsChain:
    """
    Ordered chain of AudioEffect instances.
    Supports adding, removing, reordering, and bypassing effects.
    Processes audio sequentially through the chain.
    """
    def __init__(self):
        self.effects = []  # list of AudioEffect

    def add_effect(self, effect: AudioEffect, position: int = None):
        if position is None:
            self.effects.append(effect)
        else:
            self.effects.insert(position, effect)
        return self

    def remove_effect(self, name: str):
        self.effects = [e for e in self.effects if e.name != name]
        return self

    def reorder(self, new_order: list):
        """Reorder by list of effect names."""
        name_map = {e.name: e for e in self.effects}
        self.effects = [name_map[n] for n in new_order if n in name_map]
        return self

    def process(self, signal: np.ndarray, sr: int) -> np.ndarray:
        x = signal.astype(np.float32)
        for effect in self.effects:
            if not effect.bypass:
                x = effect.process(x, sr)
        return x

    def get_chain_diagram(self):
        """Print ASCII block diagram of the current chain."""
        lines = ['INPUT']
        for e in self.effects:
            status = 'BYPASSED' if e.bypass else 'ACTIVE  '
            lines.append(f'  |')
            lines.append(f'  v')
            lines.append(f'[{status}] {e.__class__.__name__:<20} ({e.name})')
        lines.append('  |')
        lines.append('  v')
        lines.append('OUTPUT')
        return '\n'.join(lines)

    def __repr__(self):
        return f'EffectsChain([{chr(10).join(str(e) for e in self.effects)}])'

# Quick test
chain = EffectsChain()
chain.add_effect(SchroederReverb(room_size=0.5))
chain.add_effect(DigitalDelay(delay_ms=200))
chain.add_effect(Compressor())
print(chain.get_chain_diagram())

## Section 3: Preset Manager

Presets are stored as JSON files containing the effect class name and all serializable parameters. This allows complete chains to be saved, loaded, and shared.

In [ ]:
EFFECT_REGISTRY = {
    'SchroederReverb': SchroederReverb,
    'DigitalDelay': DigitalDelay,
    'Distortion': Distortion,
    'Compressor': Compressor,
    'ParametricEQ': ParametricEQ,
    'BitCrusher': BitCrusher,
    'Chorus': Chorus,
}

BUILTIN_PRESETS = {
    'Studio Room': [
        {'class': 'SchroederReverb', 'params': {'room_size': 0.3, 'damping': 0.6, 'wet_mix': 0.3, 'pre_delay_ms': 15.0}},
        {'class': 'ParametricEQ', 'params': {'low_shelf_gain_db': 2.0, 'high_shelf_gain_db': 1.5}},
        {'class': 'Compressor', 'params': {'threshold_db': -18.0, 'ratio': 3.0, 'makeup_gain_db': 4.0}},
    ],
    'Cathedral Hall': [
        {'class': 'SchroederReverb', 'params': {'room_size': 0.95, 'damping': 0.2, 'wet_mix': 0.7, 'pre_delay_ms': 50.0}},
        {'class': 'ParametricEQ', 'params': {'low_shelf_gain_db': 3.0, 'high_shelf_gain_db': -3.0}},
    ],
    'Vintage Tape Delay': [
        {'class': 'DigitalDelay', 'params': {'delay_ms': 375.0, 'feedback': 0.55, 'wet_mix': 0.4}},
        {'class': 'ParametricEQ', 'params': {'high_shelf_freq': 6000.0, 'high_shelf_gain_db': -4.0}},
        {'class': 'Compressor', 'params': {'threshold_db': -22.0, 'ratio': 2.5, 'makeup_gain_db': 3.0}},
    ],
    'Heavy Metal Dist': [
        {'class': 'Distortion', 'params': {'drive': 18.0, 'mode': 'hard', 'post_gain': 0.3}},
        {'class': 'ParametricEQ', 'params': {'peak2_freq': 1000.0, 'peak2_gain_db': -6.0,
                                              'peak3_freq': 3000.0, 'peak3_gain_db': 4.0}},
        {'class': 'Compressor', 'params': {'threshold_db': -12.0, 'ratio': 8.0, 'makeup_gain_db': 10.0}},
    ],
    'Lo-Fi Bedroom': [
        {'class': 'BitCrusher', 'params': {'bit_depth': 8, 'downsample': 6}},
        {'class': 'ParametricEQ', 'params': {'high_shelf_freq': 5000.0, 'high_shelf_gain_db': -8.0,
                                              'low_shelf_gain_db': 4.0}},
        {'class': 'SchroederReverb', 'params': {'room_size': 0.4, 'wet_mix': 0.25}},
    ],
    'Ambient Pad': [
        {'class': 'SchroederReverb', 'params': {'room_size': 0.85, 'damping': 0.3, 'wet_mix': 0.6}},
        {'class': 'Chorus', 'params': {'num_voices': 4, 'rate_hz': 0.8, 'depth_ms': 8.0, 'wet_mix': 0.4}},
        {'class': 'ParametricEQ', 'params': {'low_shelf_gain_db': -2.0, 'high_shelf_gain_db': 2.0}},
    ],
    'Drum Bus': [
        {'class': 'Compressor', 'params': {'threshold_db': -15.0, 'ratio': 6.0,
                                            'attack_ms': 5.0, 'release_ms': 50.0, 'makeup_gain_db': 8.0}},
        {'class': 'ParametricEQ', 'params': {'low_shelf_freq': 100.0, 'low_shelf_gain_db': 3.0,
                                              'peak2_freq': 5000.0, 'peak2_gain_db': 2.0}},
        {'class': 'Distortion', 'params': {'drive': 1.5, 'mode': 'soft', 'post_gain': 0.8}},
    ],
}

class PresetManager:
    def __init__(self, preset_dir=PRESET_DIR):
        self.preset_dir = Path(preset_dir)
        self.preset_dir.mkdir(parents=True, exist_ok=True)
        self._save_builtin_presets()

    def _save_builtin_presets(self):
        for name, chain_def in BUILTIN_PRESETS.items():
            fname = self.preset_dir / (name.replace(' ', '_').lower() + '.json')
            with open(fname, 'w') as f:
                json.dump({'name': name, 'chain': chain_def}, f, indent=2)

    def save_preset(self, name: str, chain: EffectsChain):
        chain_def = []
        for e in chain.effects:
            chain_def.append({'class': e.__class__.__name__, 'params': e.get_params()})
        fname = self.preset_dir / (name.replace(' ', '_').lower() + '.json')
        with open(fname, 'w') as f:
            json.dump({'name': name, 'chain': chain_def}, f, indent=2)
        print(f'Saved preset: {fname}')

    def load_preset(self, name: str) -> EffectsChain:
        fname = self.preset_dir / (name.replace(' ', '_').lower() + '.json')
        with open(fname) as f:
            data = json.load(f)
        return self.apply_preset(data['chain'])

    def apply_preset(self, chain_def: list) -> EffectsChain:
        chain = EffectsChain()
        for item in chain_def:
            cls = EFFECT_REGISTRY.get(item['class'])
            if cls:
                effect = cls(**item['params'])
                chain.add_effect(effect)
        return chain

    def list_presets(self):
        return [p.stem.replace('_', ' ').title() for p in self.preset_dir.glob('*.json')]

pm = PresetManager()
print('Built-in presets saved:')
for p in pm.list_presets():
    print(f'  {p}')

## Section 4: Signal Loading and Fallback

In [ ]:
def load_audio(path):
    data, sr = sf.read(str(path), dtype='float32')
    if data.ndim > 1: data = data[:, 0]
    return data, sr

def get_available_sources():
    wavs = list(RAW_DIR.glob('dry_*.wav'))
    if not wavs:
        # Generate fallback signals
        t = np.linspace(0, 2.0, int(SR*2), endpoint=False)
        signals = {
            'dry_sine_440': (0.7*np.sin(2*np.pi*440*t)).astype(np.float32),
            'dry_sawtooth': None
        }
        wave = np.zeros_like(t)
        for k in range(1, 20):
            if k*220 < SR/2:
                wave += ((-1)**(k+1)/k)*np.sin(2*np.pi*k*220*t)
        signals['dry_sawtooth'] = (wave/np.max(np.abs(wave)+1e-9)*0.7).astype(np.float32)
        for name, sig in signals.items():
            if sig is not None:
                sf.write(str(RAW_DIR/f'{name}.wav'), sig, SR, subtype='PCM_16')
        wavs = list(RAW_DIR.glob('dry_*.wav'))
    return {p.stem: p for p in wavs}

sources = get_available_sources()
print(f'Available sources ({len(sources)}):')
for name in list(sources.keys())[:5]:
    print(f'  {name}')

## Section 5: Interactive Widget UI

Full ipywidgets interface: source selector, preset picker, per-effect parameter sliders, process/play/export buttons.

In [ ]:
# Current state
current_state = {
    'source': None,
    'chain': EffectsChain(),
    'output': None,
    'sr': SR
}

# === Source selector ===
source_names = list(sources.keys())
w_source = widgets.Dropdown(
    options=source_names,
    value=source_names[0] if source_names else None,
    description='Source:',
    style={'description_width': '80px'},
    layout=widgets.Layout(width='300px')
)

# === Preset selector ===
preset_names = list(BUILTIN_PRESETS.keys())
w_preset = widgets.Dropdown(
    options=['(None)'] + preset_names,
    value='(None)',
    description='Preset:',
    style={'description_width': '80px'},
    layout=widgets.Layout(width='300px')
)

# === Per-effect sliders ===
# SchroederReverb
w_rev_bypass = widgets.Checkbox(value=False, description='Reverb Enable')
w_rev_room = widgets.FloatSlider(value=0.5, min=0.1, max=1.0, step=0.05, description='Room Size', layout=widgets.Layout(width='350px'))
w_rev_damp = widgets.FloatSlider(value=0.5, min=0.0, max=1.0, step=0.05, description='Damping', layout=widgets.Layout(width='350px'))
w_rev_mix = widgets.FloatSlider(value=0.4, min=0.0, max=1.0, step=0.05, description='Wet Mix', layout=widgets.Layout(width='350px'))

# DigitalDelay
w_dly_bypass = widgets.Checkbox(value=False, description='Delay Enable')
w_dly_ms = widgets.FloatSlider(value=250, min=10, max=1000, step=10, description='Delay (ms)', layout=widgets.Layout(width='350px'))
w_dly_fb = widgets.FloatSlider(value=0.4, min=0.0, max=0.95, step=0.05, description='Feedback', layout=widgets.Layout(width='350px'))
w_dly_mix = widgets.FloatSlider(value=0.3, min=0.0, max=1.0, step=0.05, description='Wet Mix', layout=widgets.Layout(width='350px'))

# Distortion
w_dist_bypass = widgets.Checkbox(value=True, description='Distortion Enable')  # bypassed by default
w_dist_drive = widgets.FloatSlider(value=5.0, min=1.0, max=30.0, step=0.5, description='Drive', layout=widgets.Layout(width='350px'))
w_dist_post = widgets.FloatSlider(value=0.5, min=0.1, max=1.0, step=0.05, description='Post Gain', layout=widgets.Layout(width='350px'))
w_dist_mode = widgets.ToggleButtons(options=['soft', 'hard', 'asymmetric'], description='Mode', value='soft')

# Compressor
w_comp_bypass = widgets.Checkbox(value=False, description='Compressor Enable')
w_comp_thresh = widgets.FloatSlider(value=-20, min=-60, max=0, step=1, description='Threshold (dB)', layout=widgets.Layout(width='350px'))
w_comp_ratio = widgets.FloatSlider(value=4.0, min=1.0, max=20.0, step=0.5, description='Ratio', layout=widgets.Layout(width='350px'))
w_comp_makeup = widgets.FloatSlider(value=6, min=0, max=24, step=1, description='Makeup (dB)', layout=widgets.Layout(width='350px'))

# Buttons
w_process_btn = widgets.Button(description='Process', button_style='primary', icon='play')
w_export_btn = widgets.Button(description='Export WAV', button_style='success', icon='download')

# Output areas
w_out_plot = widgets.Output()
w_out_audio = widgets.Output()
w_out_status = widgets.Output()

def build_chain_from_widgets():
    chain = EffectsChain()
    # Reverb
    rev = SchroederReverb(room_size=w_rev_room.value, damping=w_rev_damp.value, wet_mix=w_rev_mix.value)
    rev.bypass = not w_rev_bypass.value
    chain.add_effect(rev)
    # Delay
    dly = DigitalDelay(delay_ms=w_dly_ms.value, feedback=w_dly_fb.value, wet_mix=w_dly_mix.value)
    dly.bypass = not w_dly_bypass.value
    chain.add_effect(dly)
    # Distortion
    dst = Distortion(drive=w_dist_drive.value, mode=w_dist_mode.value, post_gain=w_dist_post.value)
    dst.bypass = not w_dist_bypass.value
    chain.add_effect(dst)
    # Compressor
    cmp = Compressor(threshold_db=w_comp_thresh.value, ratio=w_comp_ratio.value, makeup_gain_db=w_comp_makeup.value)
    cmp.bypass = not w_comp_bypass.value
    chain.add_effect(cmp)
    return chain

def on_preset_change(change):
    preset = change['new']
    if preset == '(None)': return
    # Extract first effects for sliders
    chain_def = BUILTIN_PRESETS.get(preset, [])
    for item in chain_def:
        cls = item['class']; p = item['params']
        if cls == 'SchroederReverb':
            w_rev_bypass.value = True
            if 'room_size' in p: w_rev_room.value = p['room_size']
            if 'damping' in p: w_rev_damp.value = p['damping']
            if 'wet_mix' in p: w_rev_mix.value = p['wet_mix']
        elif cls == 'DigitalDelay':
            w_dly_bypass.value = True
            if 'delay_ms' in p: w_dly_ms.value = p['delay_ms']
            if 'feedback' in p: w_dly_fb.value = p['feedback']
            if 'wet_mix' in p: w_dly_mix.value = p['wet_mix']
        elif cls == 'Distortion':
            w_dist_bypass.value = True
            if 'drive' in p: w_dist_drive.value = p['drive']
            if 'mode' in p: w_dist_mode.value = p['mode']
            if 'post_gain' in p: w_dist_post.value = p['post_gain']
        elif cls == 'Compressor':
            w_comp_bypass.value = True
            if 'threshold_db' in p: w_comp_thresh.value = p['threshold_db']
            if 'ratio' in p: w_comp_ratio.value = p['ratio']
            if 'makeup_gain_db' in p: w_comp_makeup.value = p['makeup_gain_db']

w_preset.observe(on_preset_change, names='value')

def on_process(b):
    w_out_plot.clear_output()
    w_out_audio.clear_output()
    w_out_status.clear_output()
    src_name = w_source.value
    if src_name is None:
        with w_out_status: print('No source selected'); return
    src_path = sources.get(src_name)
    if src_path is None:
        with w_out_status: print(f'Source not found: {src_name}'); return
    dry, sr = load_audio(src_path)
    chain = build_chain_from_widgets()
    output = chain.process(dry, sr)
    current_state['output'] = output
    current_state['sr'] = sr
    with w_out_status:
        print(chain.get_chain_diagram())
    with w_out_plot:
        fig, axes = plt.subplots(1, 2, figsize=(12, 3))
        t = np.arange(min(len(dry), int(0.1*sr))) / sr * 1000
        n_show = len(t)
        axes[0].plot(t, dry[:n_show], color='steelblue', linewidth=0.8)
        axes[0].set_title('Dry (first 100ms)', fontsize=9)
        axes[0].set_xlabel('ms'); axes[0].set_ylabel('Amplitude'); axes[0].grid(True, alpha=0.3)
        axes[1].plot(t, output[:n_show], color='tomato', linewidth=0.8)
        axes[1].set_title('Processed (first 100ms)', fontsize=9)
        axes[1].set_xlabel('ms'); axes[1].grid(True, alpha=0.3)
        plt.tight_layout(); plt.show()
        # Spectrogram
        fig2, axes2 = plt.subplots(1, 2, figsize=(12, 4))
        for ax, sig_data, label in [(axes2[0], dry, 'Dry'), (axes2[1], output, 'Processed')]:
            import librosa, librosa.display
            D = np.abs(librosa.stft(sig_data, n_fft=1024, hop_length=256))
            D_db = librosa.amplitude_to_db(D, ref=np.max)
            img = librosa.display.specshow(D_db, sr=sr, hop_length=256,
                                            x_axis='time', y_axis='log', ax=ax, cmap='magma')
            ax.set_title(f'{label} Spectrogram', fontsize=9)
            plt.colorbar(img, ax=ax, format='%+2.0f dB')
        plt.tight_layout(); plt.show()
    with w_out_audio:
        display(Audio(output, rate=sr))
        print('Playback ready.')

def on_export(b):
    output = current_state.get('output')
    if output is None:
        with w_out_status: print('No output to export. Press Process first.'); return
    src = w_source.value or 'output'
    out_path = PROCESSED_DIR / f'processed_{src}.wav'
    sf.write(str(out_path), np.clip(output, -1, 1), current_state['sr'], subtype='PCM_16')
    with w_out_status:
        print(f'Exported: {out_path}')

w_process_btn.on_click(on_process)
w_export_btn.on_click(on_export)

# Layout
top_row = widgets.HBox([w_source, w_preset, w_process_btn, w_export_btn])
rev_box = widgets.VBox([widgets.HTML('<b>Reverb</b>'), w_rev_bypass, w_rev_room, w_rev_damp, w_rev_mix],
                        layout=widgets.Layout(border='1px solid #ccc', padding='8px', margin='4px'))
dly_box = widgets.VBox([widgets.HTML('<b>Delay</b>'), w_dly_bypass, w_dly_ms, w_dly_fb, w_dly_mix],
                        layout=widgets.Layout(border='1px solid #ccc', padding='8px', margin='4px'))
dist_box = widgets.VBox([widgets.HTML('<b>Distortion</b>'), w_dist_bypass, w_dist_drive, w_dist_post, w_dist_mode],
                         layout=widgets.Layout(border='1px solid #ccc', padding='8px', margin='4px'))
comp_box = widgets.VBox([widgets.HTML('<b>Compressor</b>'), w_comp_bypass, w_comp_thresh, w_comp_ratio, w_comp_makeup],
                         layout=widgets.Layout(border='1px solid #ccc', padding='8px', margin='4px'))
effects_row = widgets.HBox([rev_box, dly_box, dist_box, comp_box])
ui = widgets.VBox([top_row, effects_row, w_out_status, w_out_plot, w_out_audio])
display(ui)
print('Widget UI displayed. Select source, adjust sliders, then click Process.')

## Section 6: Live Parameter Comparison — 6-Panel Spectrogram Grid

In [ ]:
import librosa
import librosa.display

# Load or generate a reference signal
ref_src = list(sources.values())[0] if sources else None
if ref_src:
    ref_audio, ref_sr = load_audio(ref_src)
else:
    t = np.linspace(0, 2.0, int(SR*2), endpoint=False)
    ref_audio = (0.7*np.sin(2*np.pi*440*t)).astype(np.float32)
    ref_sr = SR

# Apply all 7 preset configs
configs = [('Original', ref_audio)] + [
    (name, pm.load_preset(name).process(ref_audio, ref_sr))
    for name in list(BUILTIN_PRESETS.keys())[:5]
]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, (label, audio) in zip(axes.flat, configs):
    D = np.abs(librosa.stft(audio, n_fft=1024, hop_length=256))
    D_db = librosa.amplitude_to_db(D, ref=np.max)
    img = librosa.display.specshow(D_db, sr=ref_sr, hop_length=256,
                                    x_axis='time', y_axis='log', ax=ax,
                                    cmap='magma', vmin=-80, vmax=0)
    ax.set_title(label, fontsize=9, fontweight='bold')
    ax.set_xlabel('Time (s)', fontsize=8)
    ax.set_ylabel('Hz', fontsize=8)

plt.suptitle('Preset Comparison: Spectrogram Grid', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(PROCESSED_DIR / 'plot_preset_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Preset comparison grid saved.')

In [ ]:
print('=' * 50)
print('NOTEBOOK 03 COMPLETE')
print('=' * 50)
print(f'EffectsChain: add/remove/reorder/process/bypass')
print(f'PresetManager: {len(BUILTIN_PRESETS)} built-in presets')
print(f'Widget UI: source selector, 4 effects with sliders, process/play/export')
print(f'Comparison grid: 6-panel spectrogram')
print('\nCheckmark: Notebook complete')